<a href="https://colab.research.google.com/github/Michelbet92/OmniRoute/blob/release%2Fv3.8.52/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the [ComfyUI](https://github.com/comfyanonymous/ComfyUI) repo and install the requirements.

In [1]:
!git clone -q https://github.com/comfyanonymous/ComfyUI
# Instalar PyTorch compatível com T4 (CUDA 12.1 é mais estável)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ./ComfyUI/requirements.txt

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.7/29.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.5 MB/s eta 0:00

In [4]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [ ]:
# Some Helpful Custom Nodes
!git clone -q https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/comfyui-manager

!git clone -q https://github.com/ClownsharkBatwing/RES4LYF.git /content/ComfyUI/custom_nodes/RES4LYF
!git clone -q https://github.com/rgthree/rgthree-comfy.git /content/ComfyUI/custom_nodes/rgthree-comfy
!git clone -q https://github.com/yolain/ComfyUI-Easy-Use.git /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use

Download the vae and dual text encoder needed for FLUX - (Compulsory)

In [2]:
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/cocktailpeanut/xulf-dev/resolve/main/ae.safetensors -P /content/ComfyUI/models/vae/

# 8-step distilled lora for fast inference on dev models
# !wget -q --show-progress https://huggingface.co/alimama-creative/FLUX.1-Turbo-Alpha/resolve/main/diffusion_pytorch_model.safetensors -P /content/ComfyUI/models/loras

clip_l.safetensors  100%[===================>] 234.74M   119MB/s    in 2.0s    
t5xxl_fp8_e4m3fn.sa 100%[===================>]   4.56G   191MB/s    in 33s     
ae.safetensors      100%[===================>] 319.77M  89.8MB/s    in 3.6s    



The **FP8 version** runs smoothly on the free tier (without LoRA), but if you want to use it with a LoRA, it requires more resources than the free Colab tier provides.  
In that case, you can opt for the [gguf](https://huggingface.co/collections/QuantStack/flux-ggufs-68264cfc663d50c418940b30) quantized versions or the [Nunchaku](https://huggingface.co/nunchaku-tech/models) versions instead.

>**Note** : You need only one , either a gguf quantized model (under 10GB) or the Nunchanku model

---


### Option 1 : fp8 quantized - from [Comfy-Org](https://huggingface.co/Comfy-Org/flux1-dev/tree/main)

In [3]:
# fp8 - needs more than free tier
!wget -q --show-progress https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors -P /content/ComfyUI/models/checkpoints

flux1-dev-fp8.safet 100%[===================>]  16.06G  98.2MB/s    in 2m 50s  


### Option 2 : [**gguf**](https://huggingface.co/city96/FLUX.1-dev-gguf) models.  

Git clone the [ComfyUI-GGUF](https://github.com/city96/ComfyUI-GGUF) repo and follow the instructions.

In [ ]:
# %pip install --upgrade gguf
# !rm -rf ./custom_nodes/ComfyUI-GGUF
# !git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF

In [ ]:
# Download the one you need

# flux1-dev
# !wget -q --show-progress https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-kontext-dev
# !wget -q --show-progress https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-krea-dev
# !wget -q --show-progress https://huggingface.co/QuantStack/FLUX.1-Krea-dev-GGUF/resolve/main/flux1-krea-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-schnell
# !wget -q --show-progress https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q6_K.gguf -P /content/ComfyUI/models/unet

### Option 3 : [**Nunchaku**](https://nunchaku.tech/docs/ComfyUI-nunchaku/workflows/t2i.html#nunchaku-flux-1-dev-json) models - recommended

In [ ]:
import torch
print(torch.__version__)


In [ ]:
!git clone -q https://github.com/mit-han-lab/ComfyUI-nunchaku /content/ComfyUI/custom_nodes/nunchaku_nodes
!pip install -q https://github.com/nunchaku-tech/nunchaku/releases/download/v1.0.0/nunchaku-1.0.0+torch2.8-cp312-cp312-linux_x86_64.whl

In [ ]:
# flux1-dev
!wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-dev/resolve/main/svdq-int4_r32-flux.1-dev.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-krea-dev
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-krea-dev/resolve/main/svdq-int4_r32-flux.1-krea-dev.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-schnell
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-schnell/resolve/main/svdq-int4_r32-flux.1-schnell.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-kontext-dev
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-kontext-dev/resolve/main/svdq-int4_r32-flux.1-kontext-dev.safetensors -P /content/ComfyUI/models/diffusion_models

### Run ComfyUI with ngrok

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok
import subprocess, socket, time
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

# Set ngrok token
!ngrok config add-authtoken $NGROK_TOKEN

# Start ComfyUI in background
subprocess.Popen(["python", "/content/ComfyUI/main.py", "--dont-print-server"])

# Wait until port is open
port = 8188
while True:
    try:
        sock = socket.create_connection(("127.0.0.1", port), timeout=2)
        sock.close()
        print("✅ ComfyUI server is running on port", port)
        break
    except OSError:
        print("⏳ Waiting for ComfyUI to start...")
        time.sleep(2)

# Start ngrok tunnel
public_url = ngrok.connect(8188, bind_tls=True)
print("🌐 Public URL:", public_url)


### Run ComfyUI with localtunnel

In [ ]:
# !npm install -g localtunnel

In [ ]:
# import subprocess
# import threading
# import time
# import socket
# import urllib.request
# import sys

# def wait_for_port(port):
#     while True:
#         time.sleep(0.5)
#         sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#         if sock.connect_ex(('127.0.0.1', port)) == 0:
#             sock.close()
#             break
#         sock.close()

#     print("\nComfyUI finished loading, launching localtunnel...\n")
#     print("Your public IP is:",
#           urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

#     p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
#     for line in iter(p.stdout.readline, ''):
#         sys.stdout.write(line)
#         sys.stdout.flush()

# threading.Thread(target=wait_for_port, args=(8188,), daemon=True).start()

# subprocess.run(["python", "/content/ComfyUI/main.py"])


In [9]:
import subprocess, time, socket
from google.colab.output import eval_js

log = open("/content/comfy.log", "w")
subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--dont-print-server"],
    stdout=log, stderr=subprocess.STDOUT,
)

# Espera a porta abrir (até ~120s)
for _ in range(60):
    try:
        s = socket.create_connection(("127.0.0.1", 8188), timeout=2)
        s.close()
        break
    except OSError:
        time.sleep(2)
else:
    print("❌ ComfyUI não abriu a porta. Veja o log abaixo:")
    print(open("/content/comfy.log").read()[-4000:])
    raise RuntimeError("ComfyUI falhou ao iniciar.")

# Mostra o link público
public_url = eval_js("google.colab.kernel.proxyPort(8188)")
print("Abra este link no navegador:")
print(public_url)

Abra este link no navegador:
https://8188-gpu-t4-s-kkb-ass1c2-2z2wi5c6b4bv6-c.asia-southeast1-2.prod.colab.dev
